# LLM09: LoRA Fine-Tuning - Parameter-Efficient Model Adaptation

## Lab Overview

This lab explores Low-Rank Adaptation (LoRA), a parameter-efficient fine-tuning technique for large language models.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

Master LoRA implementation and application for efficient model adaptation, including mathematical foundations, architectural integration, and performance optimization.

1. **Understand LoRA Theory**: Grasp the mathematical foundations of low-rank matrix decomposition ($\Delta W \approx AB^T$).
2. **Implement LoRA Layers**: Build `LoRALayer` and `LinearWithLoRA` from scratch with proper initialization.
3. **Apply Parameter Efficiency**: Reduce trainable parameters by orders of magnitude.
4. **Integrate with Networks**: Apply LoRA to linear layers in a network with different strategies.
5. **Analyze Trade-offs**: Understand rank/alpha selection and performance implications.

---


## 1. Environment Setup


In [22]:
# Core libraries for LoRA implementation
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.9.1+rocm7.2.0.git7e1940d4
GPU: AMD Radeon(TM) 8060S Graphics
GPU Memory: 102.7 GB


## 2. LoRA Mathematical Foundations

Low-Rank Adaptation leverages the insight that neural network adaptations often lie in low-dimensional subspaces. Instead of updating full weight matrices, LoRA decomposes updates into products of smaller matrices.

**Core Mathematical Concept:**

**Traditional Fine-tuning:**

- Update full weight matrix: `W_new = W_original + ΔW`
- Parameters to train: `d × k` (full matrix size)
- Memory requirement: Store and update entire weight matrix

**LoRA Approach:**

- Decompose update: `ΔW = A × B^T`
- Where `A ∈ R^(d×r)` and `B ∈ R^(k×r)`
- Parameters to train: `r × (d + k)` where `r << min(d,k)`
- Memory requirement: Only store and update the two low-rank factors

In code, for implementation convenience, we often store the second factor directly in its transposed form, with shape `(r, k)`, and still call it `B`.  
So the implementation may compute `x @ A @ B`, which is mathematically equivalent to `x @ A @ B^T` under a different storage convention. The difference is only notation, not the underlying LoRA method.

**Key Benefits:**

**Parameter Efficiency:**

- Reduction ratio: `(d × k) / (r × (d + k))`
- For typical values: 100-1000x parameter reduction
- Example: 4096×4096 matrix requires 16M parameters, LoRA with r=16 needs only 131K

**Mathematical Properties:**

- **Rank Control**: Parameter `r` controls adaptation expressiveness
- **Scaling Factor**: Alpha parameter `α` controls adaptation strength
- **Initialization**: Proper initialization ensures training stability
- **Composability**: Multiple LoRA adapters can be combined or switched

**Computational Advantages:**

- **Forward Pass**: `y = Wx + α(x A B^T) = Wx + α((xA)B^T)`
- **Memory**: Intermediate computation `xA` has shape `(batch, r)`
- **Efficiency**: The low-rank branch uses much fewer trainable parameters. This is especially beneficial for training memory and optimizer-state cost, while inference speed may or may not improve depending on implementation.


In [23]:
# Implement LoRA Layer from Scratch
print("Building LoRA layer with mathematical rigor")


class LoRALayer(nn.Module):
    """
    Low-Rank Adaptation layer implementing a low-rank update.
    Mathematically, LoRA is often written as ΔW = A × B^T.
    In this implementation, for convenience, we store the second factor
    directly with shape (rank, out_dim), so the computation becomes x @ A @ B.
    This is equivalent to the mathematical form under a different notation convention.

    Args:
        in_dim: Input dimension
        out_dim: Output dimension
        rank: Rank of the decomposition (r)
        alpha: Scaling factor for LoRA adaptation
        dropout: Dropout probability for regularization
    """

    def __init__(self, in_dim: int, out_dim: int, rank: int, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.in_dim = in_dim
        self.out_dim = out_dim

        # Initialize A matrix with Gaussian distribution
        # Standard deviation based on rank for stable initialization
        std_dev = 1 / math.sqrt(rank)
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)

        # Initialize B matrix with zeros (important for stable training start)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))

        # Optional dropout for regularization
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

        # Scaling factor - controls adaptation strength
        self.scaling = alpha / rank

    def forward(self, x):
        """
        Forward pass: x @ A @ B * scaling

        Note:
            In many LoRA derivations, the update is written as A @ B^T.
            Here we directly store the second factor in transposed form
            with shape (rank, out_dim), so the implemented computation is
            (x @ A) @ B. This is mathematically equivalent.
        """
        # Apply dropout to input if specified
        x_dropped = self.dropout(x)

        # Efficient computation: (x @ A) @ B
        # This avoids explicitly forming the full low-rank update matrix
        intermediate = x_dropped @ self.A  # Shape: (..., rank)
        output = intermediate @ self.B  # Shape: (..., out_dim)

        return output * self.scaling

    def get_parameter_count(self):
        """Calculate number of trainable parameters"""
        return self.rank * (self.in_dim + self.out_dim)

    def get_compression_ratio(self, original_params):
        """Calculate parameter compression ratio"""
        lora_params = self.get_parameter_count()
        return original_params / lora_params


# Test LoRA layer implementation
print("Testing LoRA layer with different configurations")

# Test configuration
in_dim, out_dim = 512, 256
batch_size, seq_len = 4, 32

# Create test input
test_input = torch.randn(batch_size, seq_len, in_dim).to(device)
print(f"Test input shape: {test_input.shape}")

# Test different ranks
ranks = [4, 16, 64]
for rank in ranks:
    lora = LoRALayer(in_dim, out_dim, rank=rank, alpha=16.0).to(device)
    output = lora(test_input)

    original_params = in_dim * out_dim
    lora_params = lora.get_parameter_count()
    compression = lora.get_compression_ratio(original_params)

    print(f"\nRank {rank}:")
    print(f"  Output shape: {output.shape}")
    print(f"  LoRA parameters: {lora_params:,}")
    print(f"  Original parameters: {original_params:,}")
    print(f"  Compression ratio: {compression:.1f}x")
    print(f"  Output statistics: mean={output.mean().item():.4f}, std={output.std().item():.4f}")

print("\nLoRA layer implementation complete with proper initialization and scaling")

Building LoRA layer with mathematical rigor
Testing LoRA layer with different configurations
Test input shape: torch.Size([4, 32, 512])

Rank 4:
  Output shape: torch.Size([4, 32, 256])
  LoRA parameters: 3,072
  Original parameters: 131,072
  Compression ratio: 42.7x
  Output statistics: mean=0.0000, std=0.0000

Rank 16:
  Output shape: torch.Size([4, 32, 256])
  LoRA parameters: 12,288
  Original parameters: 131,072
  Compression ratio: 10.7x
  Output statistics: mean=0.0000, std=0.0000

Rank 64:
  Output shape: torch.Size([4, 32, 256])
  LoRA parameters: 49,152
  Original parameters: 131,072
  Compression ratio: 2.7x
  Output statistics: mean=0.0000, std=0.0000

LoRA layer implementation complete with proper initialization and scaling


## 3. LoRA Integration with Existing Layers

Now we'll create a wrapper that integrates LoRA with existing linear layers, enabling parameter-efficient adaptation of pre-trained models.

**Integration Strategy:**

**Additive Adaptation:**

- Original computation: `y = xW^T + b`
- LoRA adds a low-rank update to the weight matrix: `ΔW = AB^T`
- Final form: `y = x(W + αΔW)^T + b = xW^T + b + α(xA)B^T`

For implementation convenience, the code below stores $B^T$ directly as `B`, so `x @ A @ B` is equivalent to the usual mathematical form $xAB^T$ under a different parameter naming convention.

**Design Principles:**

- **Frozen Base**: Original layer parameters remain unchanged
- **Additive Updates**: LoRA output is added to original output
- **Selective Application**: Apply LoRA only to chosen layers
- **Multiple Adapters**: Support for multiple task-specific adaptations

**Implementation Considerations:**

- **Parameter Freezing**: Ensure base model parameters don't update
- **Gradient Flow**: LoRA parameters receive gradients, base parameters don't
- **Memory Efficiency**: Avoid storing large intermediate matrices
- **Computational Efficiency**: Optimize matrix multiplication order


In [24]:
# Implement LoRA-Enhanced Linear Layer
print("Creating LinearWithLoRA for seamless integration")


class LinearWithLoRA(nn.Module):
    """
    Linear layer enhanced with LoRA adaptation.
    Combines a frozen base linear layer with a trainable low-rank branch.
    """

    def __init__(self, linear_layer: nn.Linear, rank: int, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()

        self.linear = linear_layer
        self.lora = LoRALayer(
            in_dim=linear_layer.in_features,
            out_dim=linear_layer.out_features,
            rank=rank,
            alpha=alpha,
            dropout=dropout,
        )

        # Freeze the original linear layer by default
        self.freeze_original_parameters()

    def freeze_original_parameters(self):
        """Freeze original linear layer parameters"""
        for param in self.linear.parameters():
            param.requires_grad = False

    def unfreeze_original_parameters(self):
        """Unfreeze original linear layer parameters (for comparison only)"""
        for param in self.linear.parameters():
            param.requires_grad = True

    def forward(self, x):
        base_output = self.linear(x)
        lora_output = self.lora(x)
        return base_output + lora_output

    def get_parameter_analysis(self):
        """Analyze parameter counts and trainable-parameter savings"""
        original_params = sum(p.numel() for p in self.linear.parameters())
        original_trainable = sum(p.numel() for p in self.linear.parameters() if p.requires_grad)

        lora_params = sum(p.numel() for p in self.lora.parameters())
        lora_trainable = sum(p.numel() for p in self.lora.parameters() if p.requires_grad)

        total_params = original_params + lora_params
        total_trainable = original_trainable + lora_trainable

        # Compare against full fine-tuning of the original linear layer
        full_ft_trainable = original_params

        return {
            "original_params": original_params,
            "original_trainable": original_trainable,
            "lora_params": lora_params,
            "lora_trainable": lora_trainable,
            "total_params": total_params,
            "total_trainable": total_trainable,
            "trainable_fraction": total_trainable / total_params if total_params > 0 else 0.0,
            "trainable_reduction_vs_full_ft": full_ft_trainable / total_trainable
            if total_trainable > 0
            else float("inf"),
        }


# Demonstrate LoRA integration
print("Testing LoRA integration with existing linear layers")

torch.manual_seed(123)
original_layer = nn.Linear(10, 2).to(device)
test_input = torch.randn(1, 10).to(device)

print(f"Original layer parameters: {sum(p.numel() for p in original_layer.parameters())}")
print(f"Test input shape: {test_input.shape}")

with torch.no_grad():
    original_output = original_layer(test_input)
    print(f"Original output: {original_output}")

# Create LoRA-enhanced version
lora_layer = LinearWithLoRA(original_layer, rank=2, alpha=4.0).to(device)

analysis = lora_layer.get_parameter_analysis()
print("\nParameter Analysis:")
print(f"  Original parameters: {analysis['original_params']} (trainable: {analysis['original_trainable']})")
print(f"  LoRA parameters: {analysis['lora_params']} (trainable: {analysis['lora_trainable']})")
print(f"  Total parameters after adding LoRA: {analysis['total_params']}")
print(f"  Total trainable parameters: {analysis['total_trainable']}")
print(f"  Trainable fraction: {analysis['trainable_fraction'] * 100:.2f}%")
print(f"  Trainable reduction vs full fine-tuning: {analysis['trainable_reduction_vs_full_ft']:.1f}x")

# At initialization, outputs should match exactly because B is all zeros
with torch.no_grad():
    lora_output_init = lora_layer(test_input)

print(f"\nLoRA-enhanced output at initialization: {lora_output_init}")
difference_init = (lora_output_init - original_output).abs().max().item()
print(f"Max absolute difference at initialization: {difference_init:.6f}")
print("This should be 0 (or extremely close to 0) because the LoRA branch starts at zero.")

# Simulate a post-training adapter state for demonstration only
with torch.no_grad():
    lora_layer.lora.B.normal_(mean=0.0, std=0.02)
    lora_output_after_update = lora_layer(test_input)

difference_after = (lora_output_after_update - original_output).abs().mean().item()
print(f"\nLoRA-enhanced output after simulating a learned update: {lora_output_after_update}")
print(f"Mean absolute difference after simulated update: {difference_after:.6f}")
print("A non-zero difference now shows how the LoRA branch changes the base layer output after training.")

Creating LinearWithLoRA for seamless integration
Testing LoRA integration with existing linear layers
Original layer parameters: 22
Test input shape: torch.Size([1, 10])
Original output: tensor([[0.6639, 0.4487]], device='cuda:0')

Parameter Analysis:
  Original parameters: 22 (trainable: 0)
  LoRA parameters: 24 (trainable: 24)
  Total parameters after adding LoRA: 46
  Total trainable parameters: 24
  Trainable fraction: 52.17%
  Trainable reduction vs full fine-tuning: 0.9x

LoRA-enhanced output at initialization: tensor([[0.6639, 0.4487]], device='cuda:0')
Max absolute difference at initialization: 0.000000
This should be 0 (or extremely close to 0) because the LoRA branch starts at zero.

LoRA-enhanced output after simulating a learned update: tensor([[0.5180, 0.5032]], device='cuda:0')
Mean absolute difference after simulated update: 0.100163
A non-zero difference now shows how the LoRA branch changes the base layer output after training.


In [25]:
# Demonstrate Parameter Efficiency Across Different Configurations
print("Analyzing parameter efficiency across different LoRA configurations")

configurations = [
    {"rank": 1, "alpha": 1.0},
    {"rank": 2, "alpha": 4.0},
    {"rank": 4, "alpha": 8.0},
    {"rank": 8, "alpha": 16.0},
]

# Create a reference base layer and input
torch.manual_seed(2026)
base_layer_template = nn.Linear(512, 256).to(device)
base_state = {k: v.detach().clone() for k, v in base_layer_template.state_dict().items()}
test_batch = torch.randn(8, 32, 512).to(device)

print(f"Base layer: {base_layer_template.in_features} -> {base_layer_template.out_features}")
print(f"Test input shape: {test_batch.shape}")
print()

original_params = sum(p.numel() for p in base_layer_template.parameters())
print(f"Original layer parameters: {original_params:,}")

results = []
for config in configurations:
    # Recreate the same base layer so comparisons are fair
    base_layer = nn.Linear(512, 256).to(device)
    base_layer.load_state_dict(base_state)

    lora_enhanced = LinearWithLoRA(base_layer, rank=config["rank"], alpha=config["alpha"]).to(device)
    analysis = lora_enhanced.get_parameter_analysis()

    with torch.no_grad():
        original_out = base_layer(test_batch)
        lora_out_init = lora_enhanced(test_batch)
        init_delta = (lora_out_init - original_out).norm().item()

        # Simulate a trained adapter so we can visualize non-zero adaptation
        lora_enhanced.lora.B.normal_(mean=0.0, std=0.02)
        lora_out_after = lora_enhanced(test_batch)

        adaptation_magnitude = (lora_out_after - original_out).norm().item()
        adaptation_relative = adaptation_magnitude / original_out.norm().item()

    result = {
        **config,
        **analysis,
        "init_delta": init_delta,
        "adaptation_magnitude": adaptation_magnitude,
        "adaptation_relative": adaptation_relative,
    }
    results.append(result)

    print(f"Rank {config['rank']}, Alpha {config['alpha']}:")
    print(f"  Trainable parameters: {analysis['total_trainable']:,}")
    print(f"  Trainable reduction vs full fine-tuning: {analysis['trainable_reduction_vs_full_ft']:.1f}x")
    print(f"  Initial adaptation magnitude: {init_delta:.6f}")
    print(f"  Simulated post-training adaptation magnitude: {adaptation_magnitude:.4f}")
    print(f"  Relative adaptation: {adaptation_relative * 100:.2f}%")
    print()

print("Configuration Comparison Summary:")
print("=" * 90)
print(f"{'Rank':<6} {'Alpha':<8} {'Trainable':<12} {'Reduction':<12} {'InitDelta':<12} {'Adapt%':<10}")
print("=" * 90)

for result in results:
    print(
        f"{result['rank']:<6} "
        f"{result['alpha']:<8.1f} "
        f"{result['total_trainable']:<12,} "
        f"{result['trainable_reduction_vs_full_ft']:<12.1f}x "
        f"{result['init_delta']:<12.6f} "
        f"{result['adaptation_relative'] * 100:<10.2f}%"
    )

print(
    "\nNote: 'InitDelta' is zero because B starts at zero. Non-zero adaptation appears only after LoRA parameters are updated."
)

Analyzing parameter efficiency across different LoRA configurations
Base layer: 512 -> 256
Test input shape: torch.Size([8, 32, 512])

Original layer parameters: 131,328
Rank 1, Alpha 1.0:
  Trainable parameters: 768
  Trainable reduction vs full fine-tuning: 171.0x
  Initial adaptation magnitude: 0.000000
  Simulated post-training adaptation magnitude: 125.6868
  Relative adaptation: 84.37%

Rank 2, Alpha 4.0:
  Trainable parameters: 1,536
  Trainable reduction vs full fine-tuning: 85.5x
  Initial adaptation magnitude: 0.000000
  Simulated post-training adaptation magnitude: 244.7564
  Relative adaptation: 164.29%

Rank 4, Alpha 8.0:
  Trainable parameters: 3,072
  Trainable reduction vs full fine-tuning: 42.8x
  Initial adaptation magnitude: 0.000000
  Simulated post-training adaptation magnitude: 236.5621
  Relative adaptation: 158.79%

Rank 8, Alpha 16.0:
  Trainable parameters: 6,144
  Trainable reduction vs full fine-tuning: 21.4x
  Initial adaptation magnitude: 0.000000
  Simula

### Key Insights

- Higher rank increases LoRA capacity, but also increases the number of trainable parameters.
- In standard LoRA, the effective scaling factor is \(\alpha / r\), so alpha and rank interact rather than being completely independent.
- With the common initialization used here (random \(A\), zero \(B\)), the LoRA branch starts as an exact zero update.
- LoRA improves efficiency by reducing the number of **trainable** parameters, even though the total parameter count slightly increases after adding adapter weights.


## 4. Multi-Layer Network Integration

Now we'll apply LoRA to complete neural networks, demonstrating how to selectively adapt different layers and analyze the impact on model behavior.

**Integration Strategies:**

**Selective Adaptation:**

- **All Layers**: Apply LoRA to every linear layer
- **Output Layers**: Only adapt final classification/output layers
- **Input + Output Layers**: Adapt only the first and last linear layers
- **Selective Layers**: Adapt chosen linear layers based on a simple rule or design choice

**Layer-Specific Configuration:**

- **Different Ranks**: Use varying ranks for different layer types
- **Adaptive Alpha**: Scale adaptation strength per layer
- **Targeted Adaptation**: Apply LoRA based on layer importance

**Network Architecture Considerations:**

- **Parameter Distribution**: Where are most parameters located?
- **Gradient Flow**: Which layers benefit most from adaptation?
- **Task Relevance**: Which layers are most important for target tasks?


In [26]:
# Create and Analyze Multi-Layer Network with LoRA
print("Building comprehensive multi-layer network for LoRA integration")


class MultilayerPerceptron(nn.Module):
    """
    Multi-layer perceptron for demonstrating LoRA integration strategies
    """

    def __init__(self, num_features: int, num_hidden_1: int, num_hidden_2: int, num_classes: int):
        super().__init__()

        # Define network layers
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),  # Input projection
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),  # Hidden transformation
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),  # Output projection
        )

        # Layer names for analysis
        self.layer_names = ["input_proj", "hidden_transform", "output_proj"]

    def forward(self, x):
        return self.layers(x)

    def get_linear_layers(self):
        """Extract linear layers for LoRA application"""
        linear_layers = []
        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.Linear):
                linear_layers.append((i, layer))
        return linear_layers

    def analyze_parameters(self):
        """Analyze parameter distribution across layers"""
        analysis = {}
        total_params = 0

        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.Linear):
                layer_params = sum(p.numel() for p in layer.parameters())
                layer_name = self.layer_names[i // 2]  # Account for ReLU layers
                analysis[layer_name] = {
                    "layer_index": i,
                    "parameters": layer_params,
                    "shape": (layer.in_features, layer.out_features),
                }
                total_params += layer_params

        # Add percentage information
        for layer_info in analysis.values():
            layer_info["percentage"] = 100 * layer_info["parameters"] / total_params

        analysis["total_parameters"] = total_params
        return analysis


# Create test network
print("Creating multi-layer perceptron")
model = MultilayerPerceptron(num_features=100, num_hidden_1=200, num_hidden_2=300, num_classes=50).to(device)

print("Network architecture:")
for i, (name, module) in enumerate(model.named_modules()):
    if isinstance(module, nn.Linear):
        print(f"  {name}: {module.in_features} -> {module.out_features}")

# Analyze original network
original_analysis = model.analyze_parameters()
print("\nOriginal network parameter analysis:")
print(f"Total parameters: {original_analysis['total_parameters']:,}")

for layer_name, info in original_analysis.items():
    if isinstance(info, dict) and "parameters" in info:
        print(f"  {layer_name}: {info['parameters']:,} params ({info['percentage']:.1f}%) - {info['shape']}")

# Create test data
batch_size = 16
test_data = torch.randn(batch_size, 100).to(device)
print(f"\nTest data shape: {test_data.shape}")

# Get original output for comparison
with torch.no_grad():
    original_output = model(test_data)
    print(f"Original output shape: {original_output.shape}")
    print(
        f"Original output statistics: mean={original_output.mean().item():.4f}, std={original_output.std().item():.4f}"
    )

print("\nNetwork ready for LoRA integration")

Building comprehensive multi-layer network for LoRA integration
Creating multi-layer perceptron
Network architecture:
  layers.0: 100 -> 200
  layers.2: 200 -> 300
  layers.4: 300 -> 50

Original network parameter analysis:
Total parameters: 95,550
  input_proj: 20,200 params (21.1%) - (100, 200)
  hidden_transform: 60,300 params (63.1%) - (200, 300)
  output_proj: 15,050 params (15.8%) - (300, 50)

Test data shape: torch.Size([16, 100])
Original output shape: torch.Size([16, 50])
Original output statistics: mean=-0.0069, std=0.1025

Network ready for LoRA integration


In [27]:
# Apply LoRA to Multi-Layer Network with Different Strategies
print("Implementing various LoRA integration strategies")


def apply_lora_to_network(model, strategy="all", rank=4, alpha=8.0):
    """
    Apply LoRA to network layers based on different strategies

    Args:
        model: The neural network model
        strategy: 'all', 'output_only', 'input_output', or 'selective'
                  (the examples below demonstrate the first three strategies)
        rank: LoRA rank parameter
        alpha: LoRA alpha parameter
    """
    linear_layers = model.get_linear_layers()

    if strategy == "all":
        indices_to_modify = [i for i, _ in linear_layers]
    elif strategy == "output_only":
        indices_to_modify = [linear_layers[-1][0]]
    elif strategy == "input_output":
        indices_to_modify = [linear_layers[0][0], linear_layers[-1][0]]
    elif strategy == "selective":
        indices_to_modify = []
        for i, layer in linear_layers:
            layer_size = layer.in_features * layer.out_features
            layer_rank = rank * 2 if layer_size > 10000 else rank
            indices_to_modify.append((i, layer_rank))
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    modified_count = 0
    for item in indices_to_modify:
        if isinstance(item, tuple):
            i, layer_rank = item
        else:
            i, layer_rank = item, rank

        original_layer = model.layers[i]
        lora_layer = LinearWithLoRA(original_layer, rank=layer_rank, alpha=alpha).to(device)
        model.layers[i] = lora_layer
        modified_count += 1

    return modified_count


# Test different LoRA application strategies
strategies = ["output_only", "input_output", "all"]

for strategy in strategies:
    print(f"\n{'=' * 50}")
    print(f"Testing strategy: {strategy.upper()}")
    print(f"{'=' * 50}")

    # Create a fresh base model
    test_model = MultilayerPerceptron(100, 200, 300, 50).to(device)

    # Save the base output BEFORE adding LoRA so the comparison is fair
    with torch.no_grad():
        base_output = test_model(test_data)

    # Apply LoRA
    modified_layers = apply_lora_to_network(test_model, strategy=strategy, rank=4, alpha=8.0)
    print(f"Modified {modified_layers} layers with LoRA")

    # Simulate trained adapters for demonstration only
    with torch.no_grad():
        for module in test_model.modules():
            if isinstance(module, LinearWithLoRA):
                module.lora.B.normal_(mean=0.0, std=0.02)

    # Analyze parameter efficiency
    total_params = sum(p.numel() for p in test_model.parameters())
    trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
    trainable_reduction_ratio = total_params / trainable_params if trainable_params > 0 else float("inf")

    print("Parameter analysis:")
    print(f"  Total parameters in model: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Trainable-parameter reduction ratio: {trainable_reduction_ratio:.1f}x")
    print(f"  Percentage trainable: {100 * trainable_params / total_params:.2f}%")

    # Test model functionality
    with torch.no_grad():
        lora_output = test_model(test_data)
        output_change = (lora_output - base_output).norm().item()
        relative_change = output_change / base_output.norm().item()

    print("Output analysis:")
    print(f"  Output change magnitude: {output_change:.4f}")
    print(f"  Relative change: {relative_change:.4f} ({relative_change * 100:.2f}%)")

    # Layer-by-layer analysis
    print("Layer details:")
    for module_name, module in test_model.named_children():
        if hasattr(module, "__len__"):  # Sequential container
            for j, layer in enumerate(module):
                if isinstance(layer, LinearWithLoRA):
                    analysis = layer.get_parameter_analysis()
                    print(
                        f"  Layer {j}: LoRA enabled - "
                        f"{analysis['lora_trainable']:,} trainable params "
                        f"({analysis['trainable_reduction_vs_full_ft']:.1f}x reduction vs full FT)"
                    )
                elif isinstance(layer, nn.Linear):
                    params = sum(p.numel() for p in layer.parameters())
                    trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
                    print(f"  Layer {j}: Standard linear - {params:,} params ({trainable} trainable)")

print(
    "\nStrategy comparison complete - different strategies trade off trainable parameter count and adaptation flexibility."
)

Implementing various LoRA integration strategies

Testing strategy: OUTPUT_ONLY
Modified 1 layers with LoRA
Parameter analysis:
  Total parameters in model: 96,950
  Trainable parameters: 81,900
  Trainable-parameter reduction ratio: 1.2x
  Percentage trainable: 84.48%
Output analysis:
  Output change magnitude: 3.3986
  Relative change: 1.1593 (115.93%)
Layer details:
  Layer 0: Standard linear - 20,200 params (20200 trainable)
  Layer 2: Standard linear - 60,300 params (60300 trainable)
  Layer 4: LoRA enabled - 1,400 trainable params (10.8x reduction vs full FT)

Testing strategy: INPUT_OUTPUT
Modified 2 layers with LoRA
Parameter analysis:
  Total parameters in model: 98,150
  Trainable parameters: 62,900
  Trainable-parameter reduction ratio: 1.6x
  Percentage trainable: 64.09%
Output analysis:
  Output change magnitude: 3.3027
  Relative change: 1.1129 (111.29%)
Layer details:
  Layer 0: LoRA enabled - 1,200 trainable params (16.8x reduction vs full FT)
  Layer 2: Standard linear

In [28]:
# Comprehensive Parameter Management and Training Analysis
print("Implementing advanced parameter management for LoRA training")


def freeze_linear_layers(model, exclude_lora=True):
    """
    Freeze model parameters while optionally keeping LoRA parameters trainable.

    Args:
        model: Neural network model
        exclude_lora: If True, keep LoRA parameters trainable
    """
    frozen_params = 0
    trainable_params = 0

    for name, param in model.named_parameters():
        is_lora_param = ".lora." in name or name.endswith(".lora.A") or name.endswith(".lora.B")

        if exclude_lora and is_lora_param:
            param.requires_grad = True
            trainable_params += param.numel()
        else:
            param.requires_grad = False
            frozen_params += param.numel()

    return frozen_params, trainable_params


def analyze_gradient_flow(model, sample_input, sample_target):
    """
    Analyze gradient flow through the model to verify LoRA training setup.
    """
    model.train()
    model.zero_grad(set_to_none=True)

    # Forward pass
    output = model(sample_input)

    # Create dummy loss
    if len(output.shape) > 1 and output.shape[-1] > 1:
        # Classification scenario
        loss = F.cross_entropy(output, sample_target)
    else:
        # Regression scenario
        loss = F.mse_loss(output, sample_target)

    # Backward pass
    loss.backward()

    # Analyze gradients
    gradient_info = {}
    total_grad_norm = 0
    params_with_grad_count = 0

    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            grad_norm = param.grad.norm().item()
            gradient_info[name] = {
                "grad_norm": grad_norm,
                "param_shape": param.shape,
                "param_count": param.numel(),
            }
            total_grad_norm += grad_norm**2
            params_with_grad_count += param.numel()

    total_grad_norm = total_grad_norm**0.5

    return {
        "loss": loss.item(),
        "total_grad_norm": total_grad_norm,
        "gradient_info": gradient_info,
        "params_with_grad_count": params_with_grad_count,
    }


# Create final model with comprehensive LoRA integration
print("Creating production-ready LoRA model")
final_model = MultilayerPerceptron(100, 200, 300, 50).to(device)

# Apply LoRA to all layers with different ranks based on layer importance
layer_configs = [
    {"layer_idx": 0, "rank": 8, "alpha": 16.0},  # Input layer - higher rank
    {"layer_idx": 2, "rank": 4, "alpha": 8.0},  # Hidden layer - medium rank
    {"layer_idx": 4, "rank": 8, "alpha": 16.0},  # Output layer - higher rank
]

print("Applying layer-specific LoRA configurations:")
for config in layer_configs:
    layer = final_model.layers[config["layer_idx"]]
    lora_layer = LinearWithLoRA(layer, rank=config["rank"], alpha=config["alpha"]).to(device)
    final_model.layers[config["layer_idx"]] = lora_layer
    print(f"  Layer {config['layer_idx']}: rank={config['rank']}, alpha={config['alpha']}")

# Freeze parameters appropriately
frozen_count, trainable_count = freeze_linear_layers(final_model, exclude_lora=True)

print("\nParameter freezing results:")
print(f"  Frozen parameters: {frozen_count:,}")
print(f"  Trainable parameters: {trainable_count:,}")
print(f"  Total parameters: {frozen_count + trainable_count:,}")
print(f"  Training efficiency: {(frozen_count + trainable_count) / trainable_count:.1f}x reduction")

# Detailed parameter analysis
print("\nDetailed parameter breakdown:")
for name, param in final_model.named_parameters():
    status = "TRAINABLE" if param.requires_grad else "FROZEN"
    print(f"  {name:<30} {str(param.shape):<20} {param.numel():<8,} {status}")

# Test gradient flow
print("\nTesting gradient flow:")
sample_input = torch.randn(4, 100).to(device)
sample_target = torch.randint(0, 50, (4,)).to(device)

gradient_analysis = analyze_gradient_flow(final_model, sample_input, sample_target)

print("Gradient flow analysis:")
print(f"  Loss value: {gradient_analysis['loss']:.6f}")
print(f"  Total gradient norm: {gradient_analysis['total_grad_norm']:.6f}")
print(f"  Parameters with gradients: {gradient_analysis['params_with_grad_count']:,}")

print("\nLoRA layer gradient details:")
for name, info in gradient_analysis["gradient_info"].items():
    if ".lora." in name:
        print(f"  {name:<25} grad_norm: {info['grad_norm']:.6f}")

# Verify training readiness
trainable_with_grads = len([name for name in gradient_analysis["gradient_info"] if ".lora." in name])

print("\nTraining readiness check:")
print(f"  LoRA parameters with gradients: {trainable_with_grads}")
print(f"  Status: {'GRADIENT FLOW VERIFIED' if trainable_with_grads > 0 else 'CHECK CONFIGURATION'}")

print("\nLoRA integration complete - model ready for parameter-efficient fine-tuning")

Implementing advanced parameter management for LoRA training
Creating production-ready LoRA model
Applying layer-specific LoRA configurations:
  Layer 0: rank=8, alpha=16.0
  Layer 2: rank=4, alpha=8.0
  Layer 4: rank=8, alpha=16.0

Parameter freezing results:
  Frozen parameters: 95,550
  Trainable parameters: 7,200
  Total parameters: 102,750
  Training efficiency: 14.3x reduction

Detailed parameter breakdown:
  layers.0.linear.weight         torch.Size([200, 100]) 20,000   FROZEN
  layers.0.linear.bias           torch.Size([200])    200      FROZEN
  layers.0.lora.A                torch.Size([100, 8]) 800      TRAINABLE
  layers.0.lora.B                torch.Size([8, 200]) 1,600    TRAINABLE
  layers.2.linear.weight         torch.Size([300, 200]) 60,000   FROZEN
  layers.2.linear.bias           torch.Size([300])    300      FROZEN
  layers.2.lora.A                torch.Size([200, 4]) 800      TRAINABLE
  layers.2.lora.B                torch.Size([4, 300]) 1,200    TRAINABLE
  layer

## Conclusions

### Technical Concepts Learned

- **Low-Rank Decomposition**: Understanding ΔW = A × B^T factorization for parameter-efficient weight updates
- **LoRA Initialization**: In this notebook, `A` is randomly initialized and `B` starts at zero, which makes the initial LoRA update zero and preserves the base layer behavior at the start
- **Parameter Freezing**: Keeping base model weights frozen while only training low-rank adaptation matrices
- **Rank and Alpha Selection**: Controlling adaptation capacity (rank) and strength (alpha/rank scaling factor)
- **Integration Strategies**: Applying LoRA selectively to different linear layers in a network (for example, output-only, input+output, or all linear layers)

### Experiment Further

- Apply LoRA to transformer attention layers (Q, K, V, O projections) and compare efficiency
- Implement QLoRA combining quantization with LoRA for extreme memory efficiency
- Compare different rank values (1, 4, 16, 64) on a text classification task
- Try multiple LoRA adapters on the same base model for multi-task learning
- Experiment with AdaLoRA for automatic rank allocation during training


---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
